In [ ]:
%load_ext autoreload
%autoreload 2
import torch
from training_functions import train, evaluate_mae_mse
from mcnn import MCNN
import torch.nn as nn
import torch.optim as optim
from dataloader_mcnn import CrowdDataset
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, ConcatDataset
import glob
import os
import numpy as np
import random
import time
import pandas as pd
import seaborn as sns
SEED = 42

In [ ]:
main_folders = [r"data\ShanghaiTech_part_A\test", 
                r"data\ShanghaiTech_part_B\test",
               r'data\NWPU-Crowd\val',
              r"data\UCF-QNRF_ECCV18\test",
               r"data\jhu_crowd_v2.0\test",
               r"data\UCF_CC_50"]
evaluations_full = []
for folder in main_folders:
    evaluations = []    
    test_img_root = os.path.join(folder, "images")
    test_map_root = os.path.join(folder, "maps")
    test_dataset = CrowdDataset(test_img_root, test_map_root, gt_downsample=1, resize=False)
    for i,(img,gt_dmap) in enumerate(test_dataset):
        evaluations.append(gt_dmap.sum().item())
    print(folder.split('\\')[-2])
    print('max:', max(evaluations))
    print('min:', min(evaluations))
    print('count:', len(evaluations))
    print('sum:', sum(evaluations))
    print('mean:', np.mean(evaluations))
    if folder == r"data\UCF_CC_50":
        ind = -1
    else:
        ind = -2
    
    bins = np.logspace(
        0 if min(evaluations) == 0 else np.log10(min(evaluations)),
        np.log10(max(evaluations)),
        15
    )
    evaluations_full.append({'dataset': folder.split('\\')[ind], 'evaluations': evaluations})
    plt.hist(evaluations, bins=bins)
    plt.title(folder.split('\\')[ind])
    plt.xlabel('Number of People')
    plt.ylabel('Frequency')
    plt.xscale('log')
    plt.show()

In [ ]:
evaluations_full_df = pd.DataFrame(evaluations_full)
evaluations_full_df = evaluations_full_df.explode('evaluations')
sns.boxplot(data=pd.DataFrame(evaluations_full_df), y='evaluations', hue='dataset')
plt.title('Distribution of People Count in Test Datasets')
plt.xlabel('Dataset')
plt.ylabel('Number of People')
plt.show()

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

main_folders = [
    r"data\ShanghaiTech_part_A\test",
    r"data\ShanghaiTech_part_B\test",
    r"data\NWPU-Crowd\val",
    r"data\UCF-QNRF_ECCV18\test",
    r"data\jhu_crowd_v2.0\test",
    r"data\UCF_CC_50"
]
names = ['ShanghaiTech A', 'ShanghaiTech B', 'NWPU-Crowd', 'UCF-QNRF', 'JHU-Crowd', 'UCF-CC-50']
for i, folder in enumerate(main_folders):
    folder = Path(folder)

    image_paths = [
        p for p in folder.rglob("*")
        if p.suffix.lower() == '.jpg'
    ]

    samples = random.sample(image_paths, 4)

    fig, axes = plt.subplots(2, 2, figsize=(6, 6), constrained_layout=True)
    fig.suptitle(names[i], fontsize=12)

    for ax, img_path in zip(axes.ravel(), samples):
        img = Image.open(img_path)
        ax.imshow(img, cmap="gray")
        ax.axis("off")
    plt.show()